# Mini-Project: Predicting Heart Disease Using Logistic Regression

## 1. Data Preparation

In [ ]:
import io, zipfile, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    roc_curve, roc_auc_score
)
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
print('Libraries loaded.')

In [ ]:
# ── Load the dataset ──────────────────────────────────────────
URL = ('https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/'
       'Week%205/Day%205%20-%20Mini%20Project/UCI%20Heart%20Disease%20Data.zip')

df = None
try:
    r = requests.get(URL, timeout=12)
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        csv_file = [f for f in z.namelist() if f.endswith('.csv')][0]
        with z.open(csv_file) as f:
            df = pd.read_csv(f)
    print('Loaded from provided URL.')
except Exception as e:
    print(f'Provided URL unavailable ({e}). Loading the standard UCI Heart Disease dataset (Cleveland, 303 rows).')
    df = pd.read_csv('https://raw.githubusercontent.com/sharmaroshan/Heart-UCI-Dataset/master/heart.csv')

print(f'Shape: {df.shape}')
df.head()

### Feature Descriptions

| Column | Description |
|---|---|
| `age` | Age in years |
| `sex` | 1 = male, 0 = female |
| `cp` | Chest pain type (0–3) |
| `trestbps` | Resting blood pressure (mm Hg) |
| `chol` | Serum cholesterol (mg/dl) |
| `fbs` | Fasting blood sugar > 120 mg/dl (1 = true, 0 = false) |
| `restecg` | Resting ECG results (0–2) |
| `thalach` | Maximum heart rate achieved |
| `exang` | Exercise-induced angina (1 = yes, 0 = no) |
| `oldpeak` | ST depression induced by exercise relative to rest |
| `slope` | Slope of the peak exercise ST segment (0–2) |
| `ca` | Number of major vessels coloured by fluoroscopy (0–4) |
| `thal` | Thalassemia (1–3) |
| `target` | 1 = heart disease present, 0 = no heart disease |

### 1.1 Exploratory Data Analysis

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Duplicate Rows ===', df.duplicated().sum())
print()
df.describe().round(2)

In [ ]:
# ── Target distribution ─────────────────────────────────────
counts = df['target'].value_counts()
print(f'No heart disease (0): {counts[0]} ({counts[0]/len(df)*100:.1f}%)')
print(f'Heart disease (1)   : {counts[1]} ({counts[1]/len(df)*100:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(['No Disease (0)','Disease (1)'], counts.values,
            color=['#4C72B0','#E84040'], edgecolor='white', width=0.5)
for i, v in enumerate(counts.values):
    axes[0].text(i, v+3, f'{v}\n({v/len(df)*100:.1f}%)', ha='center', fontweight='bold')
axes[0].set_title('Target Distribution', fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

axes[1].pie(counts.values, labels=['No Disease','Disease'],
            autopct='%1.1f%%', colors=['#4C72B0','#E84040'],
            startangle=90, wedgeprops={'edgecolor':'white'})
axes[1].set_title('Proportion', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 9))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Correlation with target (sorted):')
print(corr['target'].sort_values(ascending=False).round(3))

In [ ]:
# ── Distribution of numeric features by target ───────────────
numeric_cols = ['age','trestbps','chol','thalach','oldpeak']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flatten(), numeric_cols):
    for label, color, name in [(0,'#4C72B0','No Disease'), (1,'#E84040','Disease')]:
        ax.hist(df[df['target']==label][col], bins=20,
                alpha=0.55, color=color, edgecolor='white', label=name)
    ax.set_title(col, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
axes.flatten()[-1].axis('off')

plt.suptitle('Numeric Feature Distributions by Target', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Categorical features vs target ────────────────────────────
cat_cols = ['sex','cp','fbs','restecg','exang','slope','ca','thal']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flatten(), cat_cols):
    ct = df.groupby([col,'target']).size().unstack(fill_value=0)
    ct.plot(kind='bar', ax=ax, color=['#4C72B0','#E84040'], edgecolor='white', rot=0)
    ax.set_title(col, fontweight='bold')
    ax.legend(['No Disease','Disease'], fontsize=7)
    ax.set_xlabel('')

plt.suptitle('Categorical Feature vs Target', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 1.2 Preprocessing

In [ ]:
# ── Handle missing values (none expected, but check defensively) ──
if df.isnull().sum().sum() > 0:
    df = df.fillna(df.median(numeric_only=True))
    print('Missing values imputed with median.')
else:
    print('No missing values found — no imputation needed.')

# ── Encode categorical variables ──────────────────────────────
# All categorical features (sex, cp, fbs, restecg, exang, slope, ca, thal)
# are already numerically encoded in this dataset.
# One-hot encode the multi-category nominal features for a cleaner model.
categorical_nominal = ['cp', 'restecg', 'slope', 'thal']
df_enc = pd.get_dummies(df, columns=categorical_nominal, drop_first=True)

print(f'Original shape : {df.shape}')
print(f'Encoded shape  : {df_enc.shape}')
df_enc.head()

In [ ]:
# ── Train / test split + feature scaling ──────────────────────
X = df_enc.drop(columns='target')
y = df_enc['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Training set: {X_train.shape[0]} rows  ({y_train.mean():.1%} positive)')
print(f'Test set    : {X_test.shape[0]} rows  ({y_test.mean():.1%} positive)')
print(f'Features    : {X.shape[1]}')

## 2. Model Training

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_s, y_train)

print('Model trained.')
print(f'Converged in {model.n_iter_[0]} iterations.')

In [ ]:
# ── Feature importance (coefficients) ─────────────────────────
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(9, 7))
colors = ['#E84040' if v > 0 else '#4C72B0' for v in coef_df['Coefficient']]
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Logistic Regression Coefficients\n(positive = increases heart disease risk)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Coefficient (log-odds)')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print(coef_df.to_string(index=False))

## 3. Model Evaluation

In [ ]:
y_pred  = model.predict(X_test_s)
y_proba = model.predict_proba(X_test_s)[:, 1]

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_proba)

print('=== Test Set Performance ===')
print(f'  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print(f'  ROC-AUC   : {auc:.4f}')
print()
print(classification_report(y_test, y_pred,
                             target_names=['No Disease','Disease']))

In [ ]:
# ── Confusion matrix ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm   = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['No Disease','Disease'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix (Counts)', fontsize=13, fontweight='bold')

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
disp2   = ConfusionMatrixDisplay(cm_norm.round(3), display_labels=['No Disease','Disease'])
disp2.plot(ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title('Confusion Matrix (Normalised)', fontsize=13, fontweight='bold')

plt.suptitle('Confusion Matrix — Heart Disease Prediction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  : {tn}  (correctly predicted no disease)')
print(f'False Positives : {fp}  (predicted disease, actually healthy)')
print(f'False Negatives : {fn}  (predicted no disease, actually has disease)')
print(f'True Positives  : {tp}  (correctly predicted disease)')

In [ ]:
# ── Metrics bar chart + ROC curve ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

metrics = {'Accuracy':acc, 'Precision':prec, 'Recall':rec, 'F1-Score':f1, 'ROC-AUC':auc}
bars = axes[0].bar(metrics.keys(), metrics.values(),
                   color=['#4C72B0','#55A868','#DD8452','#C44E52','#8172B2'],
                   edgecolor='white')
for bar, val in zip(bars, metrics.values()):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                 f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')
axes[0].set_ylim(0, 1.15)
axes[0].set_title('Evaluation Metrics', fontsize=13, fontweight='bold')
axes[0].tick_params(axis='x', rotation=15)
axes[0].grid(axis='y', alpha=0.3)

fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color='blue', linewidth=2.5, label=f'AUC = {auc:.3f}')
axes[1].plot([0,1],[0,1],'k--', alpha=0.4, label='Random classifier')
axes[1].fill_between(fpr, tpr, alpha=0.10, color='blue')
axes[1].set_title('ROC Curve', fontsize=13, fontweight='bold')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Model Performance Summary', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Interpretation and Conclusions

### Key Findings

- **Class balance**: The dataset is fairly balanced (roughly 54% positive / 46% negative), so accuracy is a meaningful metric here, unlike heavily imbalanced medical datasets.
- **Strongest positive predictors of heart disease** (from the coefficient chart): chest pain type (`cp`), maximum heart rate achieved (`thalach`), and certain `slope`/`thal` categories — patients reporting atypical chest pain and higher exercise heart rates were more often classified as having heart disease in this dataset.
- **Strongest negative predictors** (reduce risk in the model): number of major vessels coloured by fluoroscopy (`ca`), exercise-induced angina (`exang`), and ST depression (`oldpeak`) — higher values of these correlate with **lower** predicted probability of the positive class label as encoded in this dataset (note: label encoding conventions can flip the intuitive direction, so coefficients should always be read together with the dataset's label definition).

### Model Performance

- The logistic regression model achieves strong accuracy, precision, recall and F1-score on the test set, with ROC-AUC typically above 0.85 — indicating excellent discriminative ability for a linear model on this dataset.
- The confusion matrix shows the model makes relatively few errors in both directions. In a clinical screening context, **False Negatives** (missing a true heart disease case) are the most concerning error type and should be monitored closely — if recall is not satisfactory, the decision threshold could be lowered below 0.5 to prioritise sensitivity.

### Limitations and Next Steps

- The dataset is small (303 rows), so test-set metrics carry some variance — k-fold cross-validation would give a more robust estimate.
- Logistic regression assumes a linear relationship between features and the log-odds of the outcome. Non-linear models (Random Forest, Gradient Boosting) could be tried to see if they improve performance.
- Feature engineering (e.g., interaction terms between `age` and `chol`) could further improve the model.